In [ ]:
# librerias usadas
import pandas as pd
import numpy as np
import plotly.express as px
from imblearn.under_sampling import NearMiss
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [ ]:
# cargar la informacion limpiada
df = pd.read_csv('/content/Thechallenge2.csv')
df

,customerID,Churn,customer_gender,customer_SeniorCitizen,customer_Partner,customer_Dependents,customer_tenure,phone_PhoneService,phone_MultipleLines,internet_InternetService,...,internet_DeviceProtection,internet_TechSupport,internet_StreamingTV,internet_StreamingMovies,account_Contract,account_PaperlessBilling,account_PaymentMethod,account_Charges_Monthly,account_Charges_Total,Cuentas_Diarias
0,0002-ORFBO,0,Female,0,1,1,9,1,No,DSL,...,No,Yes,Yes,No,One year,1,Mailed check,65.60,593.30,2.186667
1,0003-MKNFE,0,Male,0,0,0,9,1,Yes,DSL,...,No,No,No,Yes,Month-to-month,0,Mailed check,59.90,542.40,1.996667
2,0004-TLHLJ,1,Male,0,0,0,4,1,No,Fiber optic,...,Yes,No,No,No,Month-to-month,1,Electronic check,73.90,280.85,2.463333
3,0011-IGKFF,1,Male,1,1,0,13,1,No,Fiber optic,...,Yes,No,Yes,Yes,Month-to-month,1,Electronic check,98.00,1237.85,3.266667
4,0013-EXCHZ,1,Female,1,1,0,3,1,No,Fiber optic,...,No,Yes,Yes,No,Month-to-month,1,Mailed check,83.90,267.40,2.796667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,9987-LUTYD,0,Female,0,0,0,13,1,No,DSL,...,No,Yes,No,No,One year,0,Mailed check,55.15,742.90,1.838333
7028,9992-RRAMN,1,Male,0,1,0,22,1,Yes,Fiber optic,...,No,No,No,Yes,Month-to-month,1,Electronic check,85.10,1873.70,2.836667
7029,9992-UJOEL,0,Male,0,0,0,2,1,No,DSL,...,No,No,No,No,Month-to-month,1,Mailed check,50.30,92.75,1.676667
7030,9993-LHIEB,0,Male,0,1,1,67,1,No,DSL,...,Yes,Yes,No,Yes,Two year,0,Mailed check,67.85,4627.65,2.261667


In [ ]:
df.columns

Index(['customerID', 'Churn', 'customer_gender', 'customer_SeniorCitizen',
       'customer_Partner', 'customer_Dependents', 'customer_tenure',
       'phone_PhoneService', 'phone_MultipleLines', 'internet_InternetService',
       'internet_OnlineSecurity', 'internet_OnlineBackup',
       'internet_DeviceProtection', 'internet_TechSupport',
       'internet_StreamingTV', 'internet_StreamingMovies', 'account_Contract',
       'account_PaperlessBilling', 'account_PaymentMethod',
       'account_Charges_Monthly', 'account_Charges_Total', 'Cuentas_Diarias'],
      dtype='object')

In [ ]:
# eliminando todas las columnas que ya se sabe que son irrelevantes
columnas_irrelevantes = ['customerID','customer_gender','phone_PhoneService',
                         'customer_SeniorCitizen','customer_Partner',
                         'customer_Dependents','phone_MultipleLines',
                         'internet_StreamingMovies','internet_StreamingTV']

In [ ]:
df = df.drop(columns=columnas_irrelevantes)

In [ ]:
df.columns

Index(['Churn', 'customer_tenure', 'internet_InternetService',
       'internet_OnlineSecurity', 'internet_OnlineBackup',
       'internet_DeviceProtection', 'internet_TechSupport', 'account_Contract',
       'account_PaperlessBilling', 'account_PaymentMethod',
       'account_Charges_Monthly', 'account_Charges_Total', 'Cuentas_Diarias'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Churn                      7032 non-null   int64  
 1   customer_tenure            7032 non-null   int64  
 2   internet_InternetService   7032 non-null   object 
 3   internet_OnlineSecurity    7032 non-null   object 
 4   internet_OnlineBackup      7032 non-null   object 
 5   internet_DeviceProtection  7032 non-null   object 
 6   internet_TechSupport       7032 non-null   object 
 7   account_Contract           7032 non-null   object 
 8   account_PaperlessBilling   7032 non-null   int64  
 9   account_PaymentMethod      7032 non-null   object 
 10  account_Charges_Monthly    7032 non-null   float64
 11  account_Charges_Total      7032 non-null   float64
 12  Cuentas_Diarias            7032 non-null   float64
dtypes: float64(3), int64(3), object(7)
memory usage:

In [ ]:
# mi tablita de valores unicos actualizada
valores_unicos = []
columnas_expandidas = list(df.columns)
for columna in columnas_expandidas:
  valores_unicos.append([columna,pd.unique(df[columna])])

df_valores_unicos = pd.DataFrame(valores_unicos, columns=['Columna', 'Valores'])
df_valores_unicos

,Columna,Valores
0,Churn,"[0, 1]"
1,customer_tenure,"[9, 4, 13, 3, 71, 63, 7, 65, 54, 72, 5, 56, 34..."
2,internet_InternetService,"[DSL, Fiber optic, No]"
3,internet_OnlineSecurity,"[No, Yes, No internet service]"
4,internet_OnlineBackup,"[Yes, No, No internet service]"
5,internet_DeviceProtection,"[No, Yes, No internet service]"
6,internet_TechSupport,"[Yes, No, No internet service]"
7,account_Contract,"[One year, Month-to-month, Two year]"
8,account_PaperlessBilling,"[1, 0]"
9,account_PaymentMethod,"[Mailed check, Electronic check, Credit card (..."


In [ ]:
# empezamos a transformar de manera numerica las variables categorica
lista_categorica = ['internet_InternetService','internet_OnlineSecurity',
                    'internet_OnlineBackup','internet_DeviceProtection',
                    'internet_TechSupport','account_Contract','account_PaymentMethod']

In [ ]:
dff = pd.get_dummies(df, columns=lista_categorica,drop_first=False)

In [ ]:
len(dff.columns) #:O

28

In [ ]:
# reaconfirmo el desvalance
dff.value_counts("Churn",normalize=True)

,proportion
Churn,
0,0.734215
1,0.265785


lo que no -> 73.42%

los que si -> 26.57%

In [ ]:
churn_pct = dff['Churn'].value_counts(normalize=True).reset_index()
churn_pct.columns = ['Churn', 'Porcentaje']
churn_pct['Porcentaje'] = churn_pct['Porcentaje'] * 100  # Convertir a porcentaje

In [ ]:
fig = px.pie(churn_pct,
             values='Porcentaje',
             names='Churn',
             title='Distribución de Churn (%)',
             color_discrete_map={0: '#2ecc71', 1: '#e74c3c'})
fig.show()

In [ ]:
fig = px.bar(churn_pct,
             x='Churn',
             y='Porcentaje',
             title='Distribución de Churn (%)',
             color='Churn',
             color_discrete_map={0: '#2ecc71', 1: '#e74c3c'},
             text=churn_pct['Porcentaje'].round(1).astype(str) + '%')
fig.show()

In [ ]:
# el submuestreo
X = dff.drop(columns='Churn')
y = dff['Churn']

nm = NearMiss()
X_resampled, y_resampled = nm.fit_resample(X, y)

# Convertir de vuelta a DataFrame
dff_r = pd.DataFrame(X_resampled, columns=X.columns)
dff_r['Churn'] = y_resampled

In [ ]:
dff_r

,customer_tenure,account_PaperlessBilling,account_Charges_Monthly,account_Charges_Total,Cuentas_Diarias,internet_InternetService_DSL,internet_InternetService_Fiber optic,internet_InternetService_No,internet_OnlineSecurity_No,internet_OnlineSecurity_No internet service,...,internet_TechSupport_No internet service,internet_TechSupport_Yes,account_Contract_Month-to-month,account_Contract_One year,account_Contract_Two year,account_PaymentMethod_Bank transfer (automatic),account_PaymentMethod_Credit card (automatic),account_PaymentMethod_Electronic check,account_PaymentMethod_Mailed check,Churn
0,1,1,69.90,69.90,2.330000,False,True,False,True,False,...,False,False,True,False,False,False,False,True,False,0
1,1,0,19.75,19.75,0.658333,False,False,True,False,True,...,True,False,True,False,False,False,False,False,True,0
2,1,0,19.75,19.75,0.658333,False,False,True,False,True,...,True,False,True,False,False,False,False,False,True,0
3,1,0,20.90,20.90,0.696667,False,False,True,False,True,...,True,False,True,False,False,False,False,False,True,0
4,1,0,20.15,20.15,0.671667,False,False,True,False,True,...,True,False,True,False,False,False,False,False,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3733,21,1,96.80,2030.30,3.226667,False,True,False,True,False,...,False,False,True,False,False,True,False,False,False,1
3734,9,1,83.85,790.15,2.795000,False,True,False,True,False,...,False,False,True,False,False,False,False,True,False,1
3735,1,1,70.15,70.15,2.338333,False,True,False,True,False,...,False,False,True,False,False,False,False,False,True,1
3736,4,0,20.95,85.50,0.698333,False,False,True,False,True,...,True,False,True,False,False,True,False,False,False,1


In [ ]:
churn_pct = dff_r['Churn'].value_counts(normalize=True).reset_index()
churn_pct.columns = ['Churn', 'Porcentaje']
churn_pct['Porcentaje'] = churn_pct['Porcentaje'] * 100  # Convertir a porcentaje

In [ ]:
fig = px.pie(churn_pct,
             values='Porcentaje',
             names='Churn',
             title='Distribución de Churn (%)',
             color_discrete_map={0: '#2ecc71', 1: '#e74c3c'})
fig.show()  #50 y 50

In [ ]:
# como la mayoria de las variables son categoricas decidi usar uno basado
# en arbol, para evitar cualquier este comportamiento inesperado decidi trasnformar
# todos esos boleanos en unos y ceros
dff_r.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3738 entries, 0 to 3737
Data columns (total 28 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   customer_tenure                                  3738 non-null   int64  
 1   account_PaperlessBilling                         3738 non-null   int64  
 2   account_Charges_Monthly                          3738 non-null   float64
 3   account_Charges_Total                            3738 non-null   float64
 4   Cuentas_Diarias                                  3738 non-null   float64
 5   internet_InternetService_DSL                     3738 non-null   bool   
 6   internet_InternetService_Fiber optic             3738 non-null   bool   
 7   internet_InternetService_No                      3738 non-null   bool   
 8   internet_OnlineSecurity_No                       3738 non-null   bool   
 9   internet_OnlineSecurity_No int

In [ ]:
lista_boleano = dff_r.select_dtypes(include='bool').columns
lista_boleano

Index(['internet_InternetService_DSL', 'internet_InternetService_Fiber optic',
       'internet_InternetService_No', 'internet_OnlineSecurity_No',
       'internet_OnlineSecurity_No internet service',
       'internet_OnlineSecurity_Yes', 'internet_OnlineBackup_No',
       'internet_OnlineBackup_No internet service',
       'internet_OnlineBackup_Yes', 'internet_DeviceProtection_No',
       'internet_DeviceProtection_No internet service',
       'internet_DeviceProtection_Yes', 'internet_TechSupport_No',
       'internet_TechSupport_No internet service', 'internet_TechSupport_Yes',
       'account_Contract_Month-to-month', 'account_Contract_One year',
       'account_Contract_Two year',
       'account_PaymentMethod_Bank transfer (automatic)',
       'account_PaymentMethod_Credit card (automatic)',
       'account_PaymentMethod_Electronic check',
       'account_PaymentMethod_Mailed check'],
      dtype='object')

In [ ]:
dff_r[lista_boleano] = dff_r[lista_boleano].astype(int)

In [ ]:
dff_r.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3738 entries, 0 to 3737
Data columns (total 28 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   customer_tenure                                  3738 non-null   int64  
 1   account_PaperlessBilling                         3738 non-null   int64  
 2   account_Charges_Monthly                          3738 non-null   float64
 3   account_Charges_Total                            3738 non-null   float64
 4   Cuentas_Diarias                                  3738 non-null   float64
 5   internet_InternetService_DSL                     3738 non-null   int64  
 6   internet_InternetService_Fiber optic             3738 non-null   int64  
 7   internet_InternetService_No                      3738 non-null   int64  
 8   internet_OnlineSecurity_No                       3738 non-null   int64  
 9   internet_OnlineSecurity_No int

In [ ]:
# la matriz de correlacion
relaciones=dff_r.corr()
relaciones.Churn

,Churn
customer_tenure,0.058315
account_PaperlessBilling,0.223738
account_Charges_Monthly,0.272793
account_Charges_Total,0.086963
Cuentas_Diarias,0.272793
internet_InternetService_DSL,-0.085420
internet_InternetService_Fiber optic,0.287218
internet_InternetService_No,-0.281228
internet_OnlineSecurity_No,0.276992
internet_OnlineSecurity_No internet service,-0.281228


In [ ]:
corr_churn = relaciones['Churn'].drop('Churn').reset_index()
corr_churn.columns = ['Variable', 'Correlación']
corr_churn = corr_churn.sort_values('Correlación')

fig = px.bar(corr_churn,
             x='Correlación',
             y='Variable',
             orientation='h',
             title='Correlación de variables numéricas con Churn',
             color='Correlación',
             color_continuous_scale=['#2ecc71', 'white', '#e74c3c'],
             range_color=[-1, 1],
             text=corr_churn['Correlación'].round(2))

fig.update_layout(
    height=800,       # aumentar si aprieta verticalmente
    yaxis=dict(tickfont=dict(size=11)),  # Tamaño de las etiquetas
    margin=dict(l=350)  # Aumenta si la etiqueta se ve cortada
)

fig.show()
# me gusto mucho esta grafica

Me agarro 10 variables



In [ ]:
var_elegidas = list(corr_churn['Variable'].head(5))

In [ ]:
var_elegidas= var_elegidas + list(corr_churn['Variable'].tail(5))

In [ ]:
var_elegidas

['internet_InternetService_No',
 'internet_DeviceProtection_No internet service',
 'internet_OnlineBackup_No internet service',
 'internet_OnlineSecurity_No internet service',
 'internet_TechSupport_No internet service',
 'Cuentas_Diarias',
 'account_Charges_Monthly',
 'internet_TechSupport_No',
 'internet_OnlineSecurity_No',
 'internet_InternetService_Fiber optic']

In [ ]:
fig1 = px.box(dff_r,
              x='Churn',
              y='customer_tenure',
              color='Churn',
              title='Tiempo de contrato vs Cancelación',
              color_discrete_map={0: '#2ecc71', 1: '#e74c3c'},
              labels={'customer_tenure': 'Meses de contrato', 'Churn': 'Cancelación'})
fig1.show()

In [ ]:
# 2. Gasto total x Cancelación (Boxplot)
fig2 = px.box(dff_r,
              x='Churn',
              y='account_Charges_Total',
              color='Churn',
              title='Gasto total vs Cancelación',
              color_discrete_map={0: '#2ecc71', 1: '#e74c3c'},
              labels={'account_Charges_Total': 'Gasto total', 'Churn': 'Cancelación'})
fig2.show()

In [ ]:
# 3. Gasto total x Tiempo de contrato x Cancelación (Scatter)
fig3 = px.scatter(dff_r,
                  x='customer_tenure',
                  y='account_Charges_Total',
                  color='Churn',
                  title='Gasto total vs Tiempo de contrato por Cancelación',
                  color_discrete_map={0: '#2ecc71', 1: '#e74c3c'},
                  labels={'customer_tenure': 'Meses de contrato',
                          'account_Charges_Total': 'Gasto total',
                          'Churn': 'Cancelación'},
                  opacity=0.6)
fig3.show()

In [ ]:
#probando el desvalanceado
relaciones=dff.corr()
relaciones.Churn

,Churn
Churn,1.000000
customer_tenure,-0.354049
account_PaperlessBilling,0.191454
account_Charges_Monthly,0.192858
account_Charges_Total,-0.199484
Cuentas_Diarias,0.192858
internet_InternetService_DSL,-0.124141
internet_InternetService_Fiber optic,0.307463
internet_InternetService_No,-0.227578
internet_OnlineSecurity_No,0.342235


In [ ]:
corr_churn = relaciones['Churn'].drop('Churn').reset_index()
corr_churn.columns = ['Variable', 'Correlación']
corr_churn = corr_churn.sort_values('Correlación')

fig = px.bar(corr_churn,
             x='Correlación',
             y='Variable',
             orientation='h',
             title='Correlación de variables numéricas con Churn',
             color='Correlación',
             color_continuous_scale=['#2ecc71', 'white', '#e74c3c'],
             range_color=[-1, 1],
             text=corr_churn['Correlación'].round(2))

fig.update_layout(
    height=800,       # aumentar si aprieta verticalmente
    yaxis=dict(tickfont=dict(size=11)),  # Tamaño de las etiquetas
    margin=dict(l=350)  # Aumenta si la etiqueta se ve cortada
)

fig.show()

In [ ]:
var_elegidas_des = list(corr_churn['Variable'].head(5))

In [ ]:
var_elegidas_des= var_elegidas_des + list(corr_churn['Variable'].tail(5))

In [ ]:
var_elegidas_des

['customer_tenure',
 'account_Contract_Two year',
 'internet_OnlineBackup_No internet service',
 'internet_InternetService_No',
 'internet_DeviceProtection_No internet service',
 'account_PaymentMethod_Electronic check',
 'internet_InternetService_Fiber optic',
 'internet_TechSupport_No',
 'internet_OnlineSecurity_No',
 'account_Contract_Month-to-month']

In [ ]:
fig1 = px.box(dff,
              x='Churn',
              y='customer_tenure',
              color='Churn',
              title='Tiempo de contrato vs Cancelación',
              color_discrete_map={0: '#2ecc71', 1: '#e74c3c'},
              labels={'customer_tenure': 'Meses de contrato', 'Churn': 'Cancelación'})
fig1.show()

In [ ]:
# 2. Gasto total x Cancelación (Boxplot)
fig2 = px.box(dff,
              x='Churn',
              y='account_Charges_Total',
              color='Churn',
              title='Gasto total vs Cancelación',
              color_discrete_map={0: '#2ecc71', 1: '#e74c3c'},
              labels={'account_Charges_Total': 'Gasto total', 'Churn': 'Cancelación'})
fig2.show()

In [ ]:
# 3. Gasto total x Tiempo de contrato x Cancelación (Scatter)
fig3 = px.scatter(dff,
                  x='customer_tenure',
                  y='account_Charges_Total',
                  color='Churn',
                  title='Gasto total vs Tiempo de contrato por Cancelación',
                  color_discrete_map={0: '#2ecc71', 1: '#e74c3c'},
                  labels={'customer_tenure': 'Meses de contrato',
                          'account_Charges_Total': 'Gasto total',
                          'Churn': 'Cancelación'},
                  opacity=0.6)
fig3.show()

In [ ]:
print(f"{var_elegidas}\n---------\n{var_elegidas_des}")

['internet_InternetService_No', 'internet_DeviceProtection_No internet service', 'internet_OnlineBackup_No internet service', 'internet_OnlineSecurity_No internet service', 'internet_TechSupport_No internet service', 'Cuentas_Diarias', 'account_Charges_Monthly', 'internet_TechSupport_No', 'internet_OnlineSecurity_No', 'internet_InternetService_Fiber optic']
---------
['customer_tenure', 'account_Contract_Two year', 'internet_OnlineBackup_No internet service', 'internet_InternetService_No', 'internet_DeviceProtection_No internet service', 'account_PaymentMethod_Electronic check', 'internet_InternetService_Fiber optic', 'internet_TechSupport_No', 'internet_OnlineSecurity_No', 'account_Contract_Month-to-month']


In [ ]:
compa = []
for val,des in zip(var_elegidas,var_elegidas_des):
  compa.append([val,dff[val].dtype,des,dff[des].dtype])

comparativa = pd.DataFrame(compa, columns=['valanceada','tipo val', 'desvalanceadas','tipo desv'])
comparativa

,valanceada,tipo val,desvalanceadas,tipo desv
0,internet_InternetService_No,bool,customer_tenure,int64
1,internet_DeviceProtection_No internet service,bool,account_Contract_Two year,bool
2,internet_OnlineBackup_No internet service,bool,internet_OnlineBackup_No internet service,bool
3,internet_OnlineSecurity_No internet service,bool,internet_InternetService_No,bool
4,internet_TechSupport_No internet service,bool,internet_DeviceProtection_No internet service,bool
5,Cuentas_Diarias,float64,account_PaymentMethod_Electronic check,bool
6,account_Charges_Monthly,float64,internet_InternetService_Fiber optic,bool
7,internet_TechSupport_No,bool,internet_TechSupport_No,bool
8,internet_OnlineSecurity_No,bool,internet_OnlineSecurity_No,bool
9,internet_InternetService_Fiber optic,bool,account_Contract_Month-to-month,bool


In [ ]:
# probando que si funcione como pienso
dff[var_elegidas]

,internet_InternetService_No,internet_DeviceProtection_No internet service,internet_OnlineBackup_No internet service,internet_OnlineSecurity_No internet service,internet_TechSupport_No internet service,Cuentas_Diarias,account_Charges_Monthly,internet_TechSupport_No,internet_OnlineSecurity_No,internet_InternetService_Fiber optic
0,False,False,False,False,False,2.186667,65.60,False,True,False
1,False,False,False,False,False,1.996667,59.90,True,True,False
2,False,False,False,False,False,2.463333,73.90,True,True,True
3,False,False,False,False,False,3.266667,98.00,True,True,True
4,False,False,False,False,False,2.796667,83.90,False,True,True
...,...,...,...,...,...,...,...,...,...,...
7027,False,False,False,False,False,1.838333,55.15,False,False,False
7028,False,False,False,False,False,2.836667,85.10,True,True,True
7029,False,False,False,False,False,1.676667,50.30,True,True,False
7030,False,False,False,False,False,2.261667,67.85,False,False,False


In [ ]:
# Dividir el original en 80/20 del cual solo voy a usar ese 20
# probare de las dos formas pr curiosidad
X_train, X_test, y_train, y_test = train_test_split(
    dff[var_elegidas_des],
    dff['Churn'],
    test_size=0.2,      # 20% prueba
    random_state=42,    # Para reproducibilidad
    stratify=dff['Churn']  # Mantiene la proporción original de Churn
)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    dff[var_elegidas],
    dff['Churn'],
    test_size=0.2,      # 20% prueba
    random_state=42,    # Para reproducibilidad
    stratify=dff['Churn']  # Mantiene la proporción original de Churn
)




In [ ]:
# Entrenamiento - del DataFrame balanceado
X_train_r = dff_r[var_elegidas]
X_train = dff_r[var_elegidas_des]
y_train = dff_r['Churn']
y_train_r = dff_r['Churn']



In [ ]:
'''
Voy a usar regresion logistica
con los dos grupos de variables elegidas
y tambien Random forest con los
dos grupos de variables
'''
# Normalizar solo para regresión logística
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modelo
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)

# Evaluación
y_pred_lr = lr.predict(X_test_scaled)
print(classification_report(y_test, y_pred_lr))


              precision    recall  f1-score   support

           0       0.88      0.71      0.79      1033
           1       0.48      0.74      0.58       374

    accuracy                           0.72      1407
   macro avg       0.68      0.72      0.68      1407
weighted avg       0.78      0.72      0.73      1407



In [ ]:
# Normalizar solo para regresión logística
scaler = StandardScaler()
X_train_scaled_r = scaler.fit_transform(X_train_r)
X_test_scaled_r = scaler.transform(X_test_r)

# Modelo
lr_r = LogisticRegression(random_state=42, max_iter=1000)
lr_r.fit(X_train_scaled_r, y_train_r)

# Evaluación rapida
y_pred_lr_r = lr_r.predict(X_test_scaled)
print(classification_report(y_test_r, y_pred_lr_r))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1033
           1       0.50      0.77      0.60       374

    accuracy                           0.73      1407
   macro avg       0.70      0.74      0.70      1407
weighted avg       0.79      0.73      0.75      1407



In [ ]:
# Modelo randomforest
rf = RandomForestClassifier(random_state=42,
    max_depth=3)
rf.fit(X_train, y_train)

# Evaluación
y_pred_rf = rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.90      0.68      0.77      1033
           1       0.47      0.79      0.59       374

    accuracy                           0.71      1407
   macro avg       0.68      0.73      0.68      1407
weighted avg       0.79      0.71      0.72      1407



In [ ]:
# Modelo randomforest
rf_r = RandomForestClassifier(random_state=42) # movele al max_depth no altero mucho los resultados
rf_r.fit(X_train_r, y_train_r)

# Evaluación
y_pred_rf_r = rf_r.predict(X_test_r)
print(classification_report(y_test_r, y_pred_rf_r))

              precision    recall  f1-score   support

           0       0.93      0.70      0.79      1033
           1       0.50      0.84      0.63       374

    accuracy                           0.74      1407
   macro avg       0.71      0.77      0.71      1407
weighted avg       0.81      0.74      0.75      1407



In [ ]:
def evaluar_modelo(nombre, y_test, y_pred):
    # Métricas
    metrics = {
        'Métrica': ['Exactitud', 'Precisión', 'Recall', 'F1-Score'],
        'Valor': [
            accuracy_score(y_test, y_pred),
            precision_score(y_test, y_pred),
            recall_score(y_test, y_pred),
            f1_score(y_test, y_pred)
        ]
    }
    df_metrics = pd.DataFrame(metrics)
    df_metrics['Valor'] = df_metrics['Valor'].round(3)

    # Gráfica de métricas
    fig1 = px.bar(df_metrics,
                  x='Métrica',
                  y='Valor',
                  title=f'Métricas — {nombre}',
                  color='Métrica',
                  text='Valor',
                  range_y=[0, 1])
    fig1.show()

    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    fig2 = px.imshow(cm,
                     text_auto=True,
                     title=f'Matriz de Confusión — {nombre}',
                     labels=dict(x='Predicción', y='Real'),
                     x=['No Canceló (0)', 'Canceló (1)'],
                     y=['No Canceló (0)', 'Canceló (1)'],
                     color_continuous_scale=['#2ecc71', '#e74c3c'])
    fig2.show()

In [ ]:
# evaluacion
evaluar_modelo('Regresión Logística [desva]', y_test, y_pred_lr)
evaluar_modelo('Regresión Logística [valan]', y_test_r, y_pred_lr_r)
evaluar_modelo('Random Forest [desva]', y_test, y_pred_rf)
evaluar_modelo('Random Forest [valan]', y_test_r, y_pred_rf_r)

Evaluacio
- el random forest desvanceado fuel el peor , 470 predijo que cancelo y se equivovo
- la regresion logistica valanceado y no valanceado tuvieron el mismo comportamineto 386 predijo que cancelo cuando no
- el random forest valanceado fue el mejor 314 predijo que cancelo cuando no y mas importante solo predijo 58 no cancelo cuando si cancelo

en si todos no parecen poder predecir si cancela o no

ninguno presento overfitting o underfitting

El campeon es Random Forest [valan] logro atrapar 316 de 374 clientes que hiban a cancelar dienso un total de 84% de hacierto
y a esos 314 que falsos positivos en el mundo real significa que le ofreces un incentivo de retención a un cliente que no iba a cancelar de todas formas. Lo peor que puede pasar es que gastes un poco más en retención, pero no pierdes al cliente.

In [ ]:
# regresion logistica
coef_lr = pd.DataFrame({
    'Variable': X_train.columns,
    'Coeficiente': lr.coef_[0]
}).sort_values('Coeficiente')

fig1 = px.bar(coef_lr,
              x='Coeficiente',
              y='Variable',
              orientation='h',
              title='Coeficientes — Regresión Logística (desvalanceado)',
              color='Coeficiente',
              color_continuous_scale=['#2ecc71', 'white', '#e74c3c'],
              range_color=[-1, 1])
fig1.update_layout(height=800, margin=dict(l=250))
fig1.show()

In [ ]:
importancia_regresion_lo_des = list(coef_lr['Variable'].head(4))

In [ ]:
# regresion logistica
coef_lr = pd.DataFrame({
    'Variable': X_train_r.columns,
    'Coeficiente': lr_r.coef_[0]
}).sort_values('Coeficiente')

fig1 = px.bar(coef_lr,
              x='Coeficiente',
              y='Variable',
              orientation='h',
              title='Coeficientes — Regresión Logística (valanceado)',
              color='Coeficiente',
              color_continuous_scale=['#2ecc71', 'white', '#e74c3c'],
              range_color=[-1, 1])
fig1.update_layout(height=800, margin=dict(l=250))
fig1.show()

In [ ]:
importancia_regresion_lo = list(coef_lr['Variable'].head(3))

In [ ]:
importancia_rf = pd.DataFrame({
    'Variable': X_train.columns,
    'Importancia': rf.feature_importances_
}).sort_values('Importancia')

fig2 = px.bar(importancia_rf,
              x='Importancia',
              y='Variable',
              orientation='h',
              title='Importancia de Variables — Random Forest (desvalanceado)',
              color='Importancia',
              color_continuous_scale=['white', '#e74c3c'])
fig2.update_layout(height=800, margin=dict(l=250))
fig2.show()

In [ ]:
imp_rf_des = list(importancia_rf['Variable'].head(2))

In [ ]:
importancia_rf = pd.DataFrame({
    'Variable': X_train_r.columns,
    'Importancia': rf_r.feature_importances_
}).sort_values('Importancia')

fig2 = px.bar(importancia_rf,
              x='Importancia',
              y='Variable',
              orientation='h',
              title='Importancia de Variables [campeon] — Random Forest (valanceado)',
              color='Importancia',
              color_continuous_scale=['white', '#e74c3c'])
fig2.update_layout(height=800, margin=dict(l=250))
fig2.show()

In [ ]:
imp_rf_val = list(importancia_rf['Variable'].head(2))

In [ ]:
print(importancia_regresion_lo)
print(importancia_regresion_lo_des)
print(imp_rf_val) #el champion
print(imp_rf_des)
# decidi agregar a tambien a cuentas diarias
originales_elegidos = ['customer_tenure','account_Contract','account_PaymentMethod','Cuentas_Diarias']
print(originales_elegidos)

['internet_InternetService_No', 'internet_DeviceProtection_No internet service', 'internet_OnlineBackup_No internet service']
['account_Contract_Two year', 'internet_OnlineBackup_No internet service', 'internet_InternetService_No', 'internet_DeviceProtection_No internet service']
['internet_OnlineBackup_No internet service', 'internet_OnlineSecurity_No internet service']
['account_Contract_Two year', 'account_Contract_Month-to-month']
['customer_tenure', 'account_Contract', 'account_PaymentMethod', 'Cuentas_Diarias']


# 📊 Informe Final

## 🎯 Objetivo
Desarrollar un modelo predictivo capaz de identificar con la mayor exactitud posible qué clientes tienen mayor probabilidad de cancelar el servicio.

---

## 🔍 Principales Hallazgos

El modelo de **Random Forest** fue el seleccionado por su mejor rendimiento, logrando detectar el **84% de los clientes que cancelaron** (Recall). Las variables con mayor peso en la predicción fueron:

| Variable | Importancia |
|---|---|
| account_Charges_Monthly | 0.405 |
| Cuentas_Diarias | 0.402 |

---

## 📌 Conclusiones

- El **monto que paga el cliente** es el factor más determinante en la cancelación del servicio.
- Los clientes con **menos meses de contrato** en realidad no afecta tanto como se habia pensado
- Los clientes con contrato **mes a mes (Month-to-month)** presentan la mayor tasa de evasión.
- El método de pago **Electronic Check** está asociado con una mayor probabilidad de abandono, posiblemente por la fricción que genera en cada transacción.

---

## ✅ Recomendaciones

1. **Descuentos por permanencia** — Ofrecer beneficios a clientes que elijan contratos anuales o bianuales en lugar del plan mes a mes.
2. **Revisión de precios** — Evaluar los paquetes con cargos mensuales más altos, ya que son los que más influyen en la cancelación.
3. **Incentivar métodos de pago automáticos** — Promover el pago con tarjeta o débito automático por encima del Electronic Check.
4. **Programa de fidelización temprana** — Enfocarse en retener clientes durante sus primeros meses, que es cuando hay mayor riesgo de cancelación.
